In [ ]:
from pathlib import Path
from tqdm.notebook import tqdm

from combra import data, angles

# Grid data generation

In [ ]:
types_dict = {
    "Ultra_Co11": "средние зерна",
    "Ultra_Co25": "мелкие зерна",
    "Ultra_Co8": "средне-мелкие зерна",
    "Ultra_Co6_2": "крупные зерна",
    "Ultra_Co15": "средне-мелкие зерна",
}

# max_img_per_class_list = [1_000, 10_000]
max_img_per_class_list = [360]
min_segment_len = 5.0
output_dir = Path("./grid_results")
output_dir.mkdir(exist_ok=True)


# sources = [
#     # san-gan
#     './data/generated_images_h5/gen_san_256x256_N100_000.h5',
#     './data/generated_images_h5/gen_san_512x512_N100_000.h5',
# ]

sources = [
    # original
    '/home/david/mnt/ssd_2_sata/python/phd/datasets/san/o_bc_left_4x_1536_1024x1024_256x256_rgb_N360',
    '/home/david/mnt/ssd_2_sata/python/phd/datasets/san/o_bc_left_4x_1536_1024x1024_512x512_rgb_N360',
    '/home/david/mnt/ssd_2_sata/python/phd/datasets/san/o_bc_left_4x_1536_1024x1024_1024x1024_rgb_N360',
]

for max_img_per_class in tqdm(max_img_per_class_list):
    for source_path in tqdm(sources):
        source_name = Path(source_path).stem
        save_path = output_dir / f"{source_name}_msl{int(min_segment_len)}"

        dataset = data.PobeditDataset(path=source_path, max_images_num_per_class=max_img_per_class)

        out = dataset.generate_angles(
            save_path=str(save_path),
            types_dict=types_dict,
            step=[0.1, 0.5, 1, 2, 3, 4, 5],
            workers=20,
            angles_tol=3,
            min_segment_len=min_segment_len,
            keep_contours=False,
            chunksize=64

        )

        print(f"[{source_name}, N={max_img_per_class}] -> {out}")

## Diffit angles generation

In [ ]:
types_dict = {
    "Ultra_Co11": "средние зерна",
    "Ultra_Co25": "мелкие зерна",
    "Ultra_Co8": "средне-мелкие зерна",
    "Ultra_Co6_2": "крупные зерна",
    "Ultra_Co15": "средне-мелкие зерна",
}

# max_img_per_class_list = [1_000, 10_000]
max_img_per_class_list = [360]
min_segment_len = 5.0
output_dir = Path("./grid_results")
output_dir.mkdir(exist_ok=True)

# Diffit-generated images: per-source N differs (256 res has 10 000, 512 res has 1 000).
diffit_sources = [
    ('./data/generated_images_h5/00017-diffit-256-gpus2-batch192_N10000.h5', [1_000]),
    ('./data/generated_images_h5/00018-diffit-512-gpus4-batch256_N1000.h5',  [1_000]),
]

for source_path, max_n_list in tqdm(diffit_sources):
    for max_n in tqdm(max_n_list):
        source_name = Path(source_path).stem
        save_path = output_dir / f"{source_name}_msl{int(min_segment_len)}"

        dataset = data.PobeditDataset(path=source_path, max_images_num_per_class=max_n)

        out = dataset.generate_angles(
            save_path=str(save_path),
            types_dict=types_dict,
            step=[0.1, 0.5, 1, 2, 3, 4, 5],
            workers=20,
            angles_tol=3,
            min_segment_len=min_segment_len,
            keep_contours=False,
            chunksize=64
        )

        print(f"[{source_name}, N={max_n}] -> {out}")

# Grid of plots

In [ ]:
# Grid plot: orig vs generated (san-gan or diffit) at each (resolution × grain class).
# Pick which generated source to overlay on top of the orig ethalon.

from combra.metrics.compare import load_rows, index_by_name_step, compute_metrics

GRID_DIR = Path('./grid_results')

# Switch between san-gan and diffit overlays.
gen_mode = 'gan'  # 'gan' or 'diffit'
# gen_mode = 'diffit' 

# Generated parquets use class_0/1/2 — same mapping for both san-gan and diffit
# (both trained on the same Ultra_Co* class ordering).
GEN_NAME_FOR = {
    'Ultra_Co25': 'class_0',
    'Ultra_Co11': 'class_1',
    'Ultra_Co6_2': 'class_2',
}

# Per-resolution orig (ethalon) folder — each resolution has its own source.
ORIG_FOLDER_FOR = {
    256: 'o_bc_left_4x_1536_1024x1024_256x256_rgb_N360_msl5',
    512: 'o_bc_left_4x_1536_1024x1024_512x512_rgb_N360_msl5',
    1024: 'o_bc_left_4x_1536_1024x1024_1024x1024_rgb_N360_msl5',
}

# San-GAN folders — only 256/512 available.
GAN_FOLDER_FOR = {
    256: 'gen_san_256x256_N100_000_msl5',
    512: 'gen_san_512x512_N100_000_msl5',
}

# Diffit folders — only 256/512 available.
DIFFIT_FOLDER_FOR = {
    256: '00017-diffit-256-gpus2-batch192_N10000_msl5',
    512: '00018-diffit-512-gpus4-batch256_N1000_msl5',
}

# Available N values per resolution per generator (limits the outer loop).
GEN_AVAILABLE_N = {
    'gan':    {256: [1_000, 10_000], 512: [1_000, 10_000]},
    'diffit': {256: [1_000, 10_000], 512: [1_000]},
}

GEN_FOLDER_FOR = {'gan': GAN_FOLDER_FOR, 'diffit': DIFFIT_FOLDER_FOR}[gen_mode]

STYLES = {
    'orig':   dict(color='blue',   marker='circle'),
    'gan':    dict(color='orange', marker='square'),
    'diffit': dict(color='green',  marker='diamond'),
}

resolutions = [256, 512, 1024]
grain_classes = [
    ('Ultra_Co25', 'мелкие зерна'),
    ('Ultra_Co11', 'средние зерна'),
    ('Ultra_Co6_2', 'крупные зерна'),
]

step = 2
save = True
ylim = [0, 0.03]

# Cache parquet -> (name, step)-indexed rows so each file is loaded once.
_rows_cache = {}
def _load_idx(parquet_path):
    if parquet_path not in _rows_cache:
        _rows_cache[parquet_path] = index_by_name_step(load_rows(parquet_path))
    return _rows_cache[parquet_path]

WDIST_COEF = 1000

for max_n in [1_000, 10_000]:
    grid = []
    metric_rows = []  # collected for printing after the grid is built
    for res in resolutions:
        row = []
        for class_key, _ in grain_classes:
            cell = []

            orig_pq = str(GRID_DIR / ORIG_FOLDER_FOR[res] / 'angles_n360.parquet')
            cell.append({
                'parquet': orig_pq,
                'class_name': f'class_{class_key}',
                'label': 'orig', **STYLES['orig'],
            })

            # Generated overlay — added only if (a) declared available in GEN_AVAILABLE_N
            # and (b) the parquet has actually been generated on disk.
            gen_pq = None
            if res in GEN_FOLDER_FOR and max_n in GEN_AVAILABLE_N[gen_mode].get(res, []):
                candidate = GRID_DIR / GEN_FOLDER_FOR[res] / f'angles_n{max_n}.parquet'
                if candidate.exists():
                    gen_pq = str(candidate)
                    cell.append({
                        'parquet': gen_pq,
                        'class_name': GEN_NAME_FOR[class_key],
                        'label': gen_mode, **STYLES[gen_mode],
                    })

            if gen_pq is not None:
                real = _load_idx(orig_pq).get((f'class_{class_key}', float(step)))
                fake = _load_idx(gen_pq).get((GEN_NAME_FOR[class_key], float(step)))
                if real is not None and fake is not None:
                    w_dist, mus_m, sig_m, amp_m = compute_metrics(real, fake)
                    metric_rows.append((res, class_key, w_dist, mus_m, sig_m, amp_m))

            row.append(cell)
        grid.append(row)

    # Print metrics block for this max_n. Skip entirely if nothing was matched
    # (e.g. diffit @ N=10000 when only N=1000 parquets exist).
    if metric_rows:
        print(f"\n=== {gen_mode}  N/class={max_n}  step={step} ===")
        header = (f"{'res':>5} {'class':<14} {'wdist*' + str(WDIST_COEF):>12}  "
                  f"{'mu1%':>7} {'mu2%':>7}  {'sig1%':>7} {'sig2%':>7}  "
                  f"{'amp1%':>7} {'amp2%':>7}")
        print(header)
        print('-' * len(header))
        for res, cls, w_dist, mus_m, sig_m, amp_m in metric_rows:
            print(f"{res:>5} {cls:<14} {w_dist*WDIST_COEF:>12.3f}  "
                  f"{mus_m[0]*100:>7.2f} {mus_m[1]*100:>7.2f}  "
                  f"{sig_m[0]*100:>7.2f} {sig_m[1]*100:>7.2f}  "
                  f"{amp_m[0]*100:>7.2f} {amp_m[1]*100:>7.2f}")

    angles.angles_plot_grid(
        grid=grid,
        row_titles=[f'{r}×{r}' for r in resolutions],
        col_titles=[label for _, label in grain_classes],
        step=step,
        ylim=ylim,
        title=f'Распределения углов ({gen_mode}, step={step}, N изобр. на класс={max_n})',
        save=f'angles_grid_{gen_mode}_n{max_n}_step{step}.png' if save else None,
    )